In [1]:
import pandas as pd
import os

def clean_high_risk_files(root_path='.'):
    print("🚑 [Phase 1] 위험 파일 4종 정밀 세척 작업 시작...")
    print("-" * 80)
    
    risk_files = [
        'RECEIVABLE_DETAILS.csv', 'RECEIVABLES.csv', 
        'PAY_BNPL_TERMS.csv', 'PAY_BNPL_REQUESTS.csv'
    ]
    
    results = []
    
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file in risk_files:
                file_path = os.path.join(root, file)
                try:
                    # 1. 원본 로드
                    df = pd.read_csv(file_path, low_memory=False, encoding='utf-8-sig')
                    original_rows = len(df)
                    
                    # 2. 식별자 컬럼 지정
                    id_col = 'COMPANY_ID'
                    if id_col not in df.columns:
                        continue
                        
                    # 3. 세척 로직 적용 (Null 제거 -> float -> int -> 10자리 문자열)
                    # NaN이 있는 행은 조인이 불가능하므로 dropna 처리
                    df_clean = df.dropna(subset=[id_col]).copy()
                    df_clean[id_col] = df_clean[id_col].astype(float).astype(int).astype(str).str.zfill(10)
                    
                    # 4. 세척된 데이터 저장 (_CLEANED.csv)
                    new_file_name = file.replace('.csv', '_CLEANED.csv')
                    new_file_path = os.path.join(root, new_file_name)
                    df_clean.to_csv(new_file_path, index=False, encoding='utf-8-sig')
                    
                    cleaned_rows = len(df_clean)
                    
                    results.append({
                        'File': file,
                        'Original_Rows': original_rows,
                        'Cleaned_Rows': cleaned_rows,
                        'Removed_Nulls': original_rows - cleaned_rows,
                        'Sample_Key_After': df_clean[id_col].iloc[0]
                    })
                    print(f"✅ 완료: {file} -> {new_file_name} 저장 성공")
                    
                except Exception as e:
                    print(f"⚠️ {file} 처리 중 에러 발생: {e}")

    # 5. 결과 리포트 출력
    if results:
        report = pd.DataFrame(results)
        print("\n📊 [Phase 1: 세척 결과 요약표]")
        display(report)
    else:
        print("\n❌ 해당 경로에서 위험 파일 4종을 찾지 못했습니다.")

# 즉시 실행
clean_high_risk_files('.')

🚑 [Phase 1] 위험 파일 4종 정밀 세척 작업 시작...
--------------------------------------------------------------------------------
✅ 완료: PAY_BNPL_REQUESTS.csv -> PAY_BNPL_REQUESTS_CLEANED.csv 저장 성공
✅ 완료: PAY_BNPL_TERMS.csv -> PAY_BNPL_TERMS_CLEANED.csv 저장 성공
✅ 완료: RECEIVABLES.csv -> RECEIVABLES_CLEANED.csv 저장 성공
✅ 완료: RECEIVABLE_DETAILS.csv -> RECEIVABLE_DETAILS_CLEANED.csv 저장 성공

📊 [Phase 1: 세척 결과 요약표]


,File,Original_Rows,Cleaned_Rows,Removed_Nulls,Sample_Key_After
0,PAY_BNPL_REQUESTS.csv,495,487,8,0000000367
1,PAY_BNPL_TERMS.csv,1729,1697,32,0000000367
2,RECEIVABLES.csv,14496,12712,1784,0000000366
3,RECEIVABLE_DETAILS.csv,14597,12800,1797,0000000095


In [3]:
import pandas as pd
import numpy as np
import os

def aggregate_dynamic_features(root_path='.'):
    print("🗜️ [Phase 2] 1:N 트랜잭션 압축(Aggregation) 가동 중...")
    print("-" * 80)
    
    # 1. 방금 생성한 CLEANED 파일 찾기
    cleaned_files = {}
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('_CLEANED.csv'):
                cleaned_files[file] = os.path.join(root, file)
                
    if not cleaned_files:
        print("❌ _CLEANED.csv 파일을 찾을 수 없습니다. Phase 1이 실행된 폴더에서 실행해주세요.")
        return None

    df_list = []

    # =========================================================
    # [압축 1] 네트워크 지표 (RECEIVABLE_DETAILS_CLEANED)
    # =========================================================
    if 'RECEIVABLE_DETAILS_CLEANED.csv' in cleaned_files:
        print("▶️ 매출채권 세부내역 압축 중 (Partner Count, Concentration)...")
        df_rec_det = pd.read_csv(cleaned_files['RECEIVABLE_DETAILS_CLEANED.csv'], dtype={'COMPANY_ID': str})
        
        # 유연한 컬럼 탐색 (환경마다 이름이 조금 다를 수 있음)
        amt_cols = [c for c in df_rec_det.columns if 'AMOUNT' in c.upper()]
        amt_col = amt_cols[0] if amt_cols else None
        
        partner_cols = [c for c in df_rec_det.columns if 'BUYER' in c.upper() or 'PARTNER' in c.upper()]
        partner_col = partner_cols[0] if partner_cols else 'COMPANY_ID'

        if amt_col:
            agg_funcs = {amt_col: ['sum', 'max']}
            agg_funcs[partner_col] = 'nunique' if partner_col != 'COMPANY_ID' else 'count'
            
            rec_agg = df_rec_det.groupby('COMPANY_ID').agg(agg_funcs).reset_index()
            rec_agg.columns = ['COMPANY_ID', 'receivable_Total_Amt', 'Max_Receivable_Amt', 'receivable_Partner_Count']
            
            # 집중도(Concentration) 계산
            rec_agg['Receivable_Concentration'] = np.where(
                rec_agg['receivable_Total_Amt'] > 0,
                rec_agg['Max_Receivable_Amt'] / rec_agg['receivable_Total_Amt'],
                0
            )
            rec_agg = rec_agg.drop(columns=['Max_Receivable_Amt'])
            df_list.append(rec_agg)
            print(f"  └─ ✅ 채권 압축 완료: {len(rec_agg)}개 기업")

    # =========================================================
    # [압축 2] BNPL 행동 지표 (PAY_BNPL_REQUESTS_CLEANED)
    # =========================================================
    if 'PAY_BNPL_REQUESTS_CLEANED.csv' in cleaned_files:
        print("▶️ BNPL 트랜잭션 압축 중 (Success Rate, Avg Amount)...")
        df_req = pd.read_csv(cleaned_files['PAY_BNPL_REQUESTS_CLEANED.csv'], dtype={'COMPANY_ID': str})
        
        amt_cols = [c for c in df_req.columns if 'AMOUNT' in c.upper()]
        amt_col = 'TOTAL_GROSS_APPLY_AMOUNT' if 'TOTAL_GROSS_APPLY_AMOUNT' in df_req.columns else (amt_cols[0] if amt_cols else None)
        
        status_cols = [c for c in df_req.columns if 'STATUS' in c.upper()]
        status_col = 'EVALUATION_STATUS' if 'EVALUATION_STATUS' in df_req.columns else (status_cols[0] if status_cols else None)
        
        if amt_col and status_col:
            req_agg = df_req.groupby('COMPANY_ID').agg(
                BNPL_Req_Count=(amt_col, 'count'),
                BNPL_Avg_Amt=(amt_col, 'mean'),
                BNPL_Success_Rate=(status_col, lambda x: x.isin(['PAYMENT', 'COMPLETE_TRADE']).mean())
            ).reset_index()
            df_list.append(req_agg)
            print(f"  └─ ✅ BNPL 압축 완료: {len(req_agg)}개 기업")

    # =========================================================
    # [최종 통합] 1:1 동적 피처 마트 병합
    # =========================================================
    if df_list:
        print("\n▶️ 압축된 피처들을 COMPANY_ID 기준으로 병합 중...")
        from functools import reduce
        dynamic_mart = reduce(lambda left, right: pd.merge(left, right, on='COMPANY_ID', how='outer'), df_list)
        
        dynamic_mart = dynamic_mart.fillna(0)
        output_file = "Dynamic_Feature_Mart_v1.csv"
        dynamic_mart.to_csv(output_file, index=False, encoding='utf-8-sig')
        
        print(f"🎉 완료! 총 {len(dynamic_mart)}개 기업의 [1:1 동적 피처 마트] 생성 ({output_file})\n")
        print("📊 [압축된 동적 피처 샘플 (Top 5)]")
        print(dynamic_mart.head().to_string(index=False))
        
        return dynamic_mart
    else:
        print("⚠️ 압축할 데이터 컬럼을 찾지 못했습니다.")
        return None

# !!! 여기서 즉시 실행됩니다 !!!
df_dynamic = aggregate_dynamic_features('.')

🗜️ [Phase 2] 1:N 트랜잭션 압축(Aggregation) 가동 중...
--------------------------------------------------------------------------------
▶️ 매출채권 세부내역 압축 중 (Partner Count, Concentration)...
  └─ ✅ 채권 압축 완료: 199개 기업
▶️ BNPL 트랜잭션 압축 중 (Success Rate, Avg Amount)...
  └─ ✅ BNPL 압축 완료: 133개 기업

▶️ 압축된 피처들을 COMPANY_ID 기준으로 병합 중...
🎉 완료! 총 321개 기업의 [1:1 동적 피처 마트] 생성 (Dynamic_Feature_Mart_v1.csv)

📊 [압축된 동적 피처 샘플 (Top 5)]
COMPANY_ID  receivable_Total_Amt  receivable_Partner_Count  Receivable_Concentration  BNPL_Req_Count  BNPL_Avg_Amt  BNPL_Success_Rate
0000000001          1258500000.0                       5.0                  0.465435             0.0           0.0                0.0
0000000002            53485300.0                       1.0                  1.000000             0.0           0.0                0.0
0000000003            15330000.0                       1.0                  1.000000             0.0           0.0                0.0
0000000004                   0.0                       0.

In [4]:
import pandas as pd
import os

def build_final_wide_table(root_path='.'):
    print("🏗️ [Phase 3] 최종 와이드 테이블(Wide-Table) 통합 가동 중...")
    print("-" * 80)
    
    # 1. 뼈대가 될 재무 및 개황 파일 찾기
    fin_file = None
    overview_file = None
    dynamic_file = "Dynamic_Feature_Mart_v1.csv"
    
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_FINANCIAL_RATIO.csv' in files:
            fin_file = os.path.join(root, 'COMPANY_FINANCIAL_RATIO.csv')
        if 'COMPANY_OVERVIEW.csv' in files:
            overview_file = os.path.join(root, 'COMPANY_OVERVIEW.csv')
            
    if not (fin_file and dynamic_file and os.path.exists(dynamic_file)):
        print("❌ 필수 파일(재무 비율 또는 앞서 만든 동적 마트)이 부족합니다.")
        return None

    # =========================================================
    # 1. 기초 마스터 구성 (1,514개 기업 뼈대 세우기)
    # =========================================================
    print("▶️ 기초 재무 마스터 정규화 및 압축 중...")
    # 개황 (Base)
    if overview_file:
        df_base = pd.read_csv(overview_file, encoding='utf-8-sig', dtype={'COMPANY_ID': str})
        df_base['COMPANY_ID'] = df_base['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
        df_base = df_base.drop_duplicates(subset=['COMPANY_ID'], keep='last')[['COMPANY_ID']]
    else:
        df_base = pd.DataFrame(columns=['COMPANY_ID'])

    # 재무 비율 (Financial Ratio)
    df_fin = pd.read_csv(fin_file, encoding='utf-8-sig', dtype={'COMPANY_ID': str})
    df_fin['COMPANY_ID'] = df_fin['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
    
    # 가장 최신 재무 데이터만 남김 (1:1 압축)
    df_fin_latest = df_fin.drop_duplicates(subset=['COMPANY_ID'], keep='last')
    
    # 필요한 핵심 재무 지표만 추출 (매뉴얼 기준)
    fin_cols = ['COMPANY_ID', 'LIQUIDITY_RATIO', 'DEBT_RATIO', 'OPERATING_PROFIT_RATIO', 'INTEREST_COVERAGE_RATIO']
    fin_cols = [c for c in fin_cols if c in df_fin_latest.columns]
    
    master = pd.merge(df_base, df_fin_latest[fin_cols], on='COMPANY_ID', how='outer')

    # =========================================================
    # 2. 동적 피처 마트 결합 (Left Join)
    # =========================================================
    print("▶️ Phase 2의 동적 피처 마트 결합 중...")
    df_dynamic = pd.read_csv(dynamic_file, dtype={'COMPANY_ID': str})
    df_dynamic['COMPANY_ID'] = df_dynamic['COMPANY_ID'].str.zfill(10)
    
    # 정적 재무(1,514) + 동적 행동(321) = 최종 마스터
    final_wide_table = pd.merge(master, df_dynamic, on='COMPANY_ID', how='left')

    # =========================================================
    # 3. 결측치 최종 보정
    # =========================================================
    # 동적 데이터가 없는 기업(Left Join의 빈칸)은 거래/행동이 '0'인 것으로 처리
    dyn_cols = [c for c in df_dynamic.columns if c != 'COMPANY_ID']
    final_wide_table[dyn_cols] = final_wide_table[dyn_cols].fillna(0)

    output_file = "Final_Integrated_Credit_Master_v1.csv"
    final_wide_table.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"\n🎉 [최종 통합 완료] 총 {len(final_wide_table)}개 기업의 평가 마스터 생성 ({output_file})")
    print(f"   └─ 포함된 Feature 개수: {final_wide_table.shape[1]}개")
    print("\n📊 [최종 와이드 테이블 샘플 (Top 5)]")
    print(final_wide_table.head().to_string(index=False))
    
    return final_wide_table

# 즉시 실행
df_final_master = build_final_wide_table('.')

🏗️ [Phase 3] 최종 와이드 테이블(Wide-Table) 통합 가동 중...
--------------------------------------------------------------------------------
▶️ 기초 재무 마스터 정규화 및 압축 중...
▶️ Phase 2의 동적 피처 마트 결합 중...

🎉 [최종 통합 완료] 총 1517개 기업의 평가 마스터 생성 (Final_Integrated_Credit_Master_v1.csv)
   └─ 포함된 Feature 개수: 7개

📊 [최종 와이드 테이블 샘플 (Top 5)]
COMPANY_ID  receivable_Total_Amt  receivable_Partner_Count  Receivable_Concentration  BNPL_Req_Count  BNPL_Avg_Amt  BNPL_Success_Rate
0000000001          1258500000.0                       5.0                  0.465435             0.0           0.0                0.0
0000000003            15330000.0                       1.0                  1.000000             0.0           0.0                0.0
0000000004                   0.0                       0.0                  0.000000             0.0           0.0                0.0
0000000005           388388000.0                       1.0                  1.000000             0.0           0.0                0.0
0000000007        

In [5]:
import pandas as pd
import os

def check_target_columns(root_path='.'):
    print("🔍 [Column Audit] 4개 분야 핵심 파일 컬럼명 스캔 중...")
    print("-" * 90)
    
    # 컬럼명을 정확히 알아야만 조인 및 피처 추출이 가능한 타겟 파일들
    files_to_check = [
        'COMPANY_FINANCIAL_RATIO.csv',             # 재무 비율 (누락 확인용)
        'COMPANY_REPRESENTATIVE_HISTORY.csv',      # 대표자 변경 (정성평가)
        'COMPANY_EMPLOYEE_STATISTICS.csv',         # 고용 지표 (정성평가)
        'COMPANY_LINKS.csv',                       # 관계사 네트워크 (정성평가)
        'PAY_BNPL_COMMENTS.csv',                   # 심사 코멘트 (심사/신뢰)
        'PAY_BNPL_UPLOAD_FILES.csv',               # 증빙 서류 (심사/신뢰)
        'PAY_BNPL_REQUEST_EVALUATION_LOGS.csv'     # 심사 로그 (심사/신뢰)
    ]
    
    found_count = 0
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file in files_to_check:
                file_path = os.path.join(root, file)
                try:
                    # nrows=0 으로 설정하여 메모리 소모 없이 컬럼명만 빠르게 추출
                    df = pd.read_csv(file_path, nrows=0, encoding='utf-8-sig')
                    cols = df.columns.tolist()
                    print(f"📄 {file}")
                    print(f"   └─ {cols}\n")
                    found_count += 1
                except Exception as e:
                    print(f"⚠️ {file} 읽기 에러: {e}")
                    
    if found_count == 0:
        print("❌ 타겟 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

# 즉시 실행
check_target_columns('.')

🔍 [Column Audit] 4개 분야 핵심 파일 컬럼명 스캔 중...
------------------------------------------------------------------------------------------
📄 COMPANY_LINKS.csv
   └─ ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'AMOUNT', 'CATEGORY', 'COMPANY_ID', 'CREATED_AT', 'DELETED_AT', 'TRADE_DATE', 'UPDATED_AT', '_AB_CDC_LSN', 'REFERENCE_ID', 'REFERENCE_TABLE', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'ANOTHER_COMPANY_ID']

📄 PAY_BNPL_COMMENTS.csv
   └─ ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'UUID', 'CONTENT', 'IS_READ', 'USER_ID', 'COMPANY_ID', 'CREATED_AT', 'DELETED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'IS_INTERNAL', 'ADMIN_USER_ID', 'EVALUATION_STAGE', 'EVALUATION_STATUS', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'PAY_BNPL_REQUEST_ID']

📄 PAY_BNPL_REQUEST_EVALUATION_LOGS.csv
   └─ ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'CREATED_AT', 'DELETED_

In [6]:
import pandas as pd
import numpy as np
import os

def build_ultimate_wide_table(root_path='.'):
    print("🏗️ [Ultimate Engine] 276홀딩스 완전체 와이드 테이블 통합 가동...")
    print("-" * 90)

    # 1. 뼈대(Base) 로드: 이전에 압축해둔 동적 마트(BNPL+매출채권)와 개황 마스터
    master = pd.DataFrame()
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_OVERVIEW.csv' in files and master.empty:
            df_base = pd.read_csv(os.path.join(root, 'COMPANY_OVERVIEW.csv'), dtype={'COMPANY_ID': str})
            df_base['COMPANY_ID'] = df_base['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            master = df_base.drop_duplicates(subset=['COMPANY_ID'], keep='last')[['COMPANY_ID']]

    # 이전에 만든 동적 마트 병합 (Phase 2 결과물)
    dynamic_file = 'Dynamic_Feature_Mart_v1.csv'
    if os.path.exists(dynamic_file):
        df_dyn = pd.read_csv(dynamic_file, dtype={'COMPANY_ID': str})
        df_dyn['COMPANY_ID'] = df_dyn['COMPANY_ID'].str.zfill(10)
        master = pd.merge(master, df_dyn, on='COMPANY_ID', how='left')

    # 2. 재무 비율 매핑 (FR_VAL 암호 해독)
    print("▶️ 재무 비율(Financial Ratio) 매핑 및 결합 중...")
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_FINANCIAL_RATIO.csv' in files:
            df_fin = pd.read_csv(os.path.join(root, 'COMPANY_FINANCIAL_RATIO.csv'), dtype={'COMPANY_ID': str})
            df_fin['COMPANY_ID'] = df_fin['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            # 최신 결산일(FR_ACCT_DT) 기준 정렬 후 최신값만 유지
            df_fin = df_fin.sort_values(by=['COMPANY_ID', 'FR_ACCT_DT']).drop_duplicates(subset=['COMPANY_ID'], keep='last')
            
            # [CIO ACTION REQUIRED] 실제 데이터 사전에 맞게 번호(FR_VAL) 수정 필요
            rename_dict = {
                'FR_VAL1': 'LIQUIDITY_RATIO',      # 유동비율 (가정)
                'FR_VAL2': 'DEBT_RATIO',           # 부채비율 (가정)
                'FR_VAL3': 'INTEREST_COVERAGE',    # 이자보상배율 (가정)
                'FR_VAL4': 'OP_MARGIN'             # 영업이익률 (가정)
            }
            df_fin = df_fin.rename(columns=rename_dict)
            cols_to_merge = ['COMPANY_ID'] + list(rename_dict.values())
            # 있는 컬럼만 병합
            cols_to_merge = [c for c in cols_to_merge if c in df_fin.columns]
            master = pd.merge(master, df_fin[cols_to_merge], on='COMPANY_ID', how='left')

    # 3. 고용 지표 및 경영진 이력 통합
    print("▶️ 조직/경영진 안정성 지표 결합 중...")
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_EMPLOYEE_STATISTICS.csv' in files:
            df_emp = pd.read_csv(os.path.join(root, 'COMPANY_EMPLOYEE_STATISTICS.csv'), dtype={'COMPANY_ID': str})
            df_emp['COMPANY_ID'] = df_emp['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            df_emp_latest = df_emp.sort_values(by=['COMPANY_ID', 'STANDARD_DATE']).drop_duplicates(subset=['COMPANY_ID'], keep='last')
            master = pd.merge(master, df_emp_latest[['COMPANY_ID', 'EMPLOYEE_COUNT']], on='COMPANY_ID', how='left')
            
        if 'COMPANY_REPRESENTATIVE_HISTORY.csv' in files:
            df_rep = pd.read_csv(os.path.join(root, 'COMPANY_REPRESENTATIVE_HISTORY.csv'), dtype={'COMPANY_ID': str})
            df_rep['COMPANY_ID'] = df_rep['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            rep_agg = df_rep.groupby('COMPANY_ID').size().reset_index(name='REP_CHANGE_COUNT')
            master = pd.merge(master, rep_agg, on='COMPANY_ID', how='left')

    # 4. 정성평가 및 신뢰도 로그 (네트워크, 코멘트, 파일 업로드)
    print("▶️ 심사 신뢰도 및 네트워크 리스크 압축 중...")
    for root, dirs, files in os.walk(root_path):
        # 파트너 링크 다변화 파악
        if 'COMPANY_LINKS.csv' in files:
            df_link = pd.read_csv(os.path.join(root, 'COMPANY_LINKS.csv'), dtype={'COMPANY_ID': str})
            df_link['COMPANY_ID'] = df_link['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            link_agg = df_link.groupby('COMPANY_ID').agg(LINKED_PARTNERS=('ANOTHER_COMPANY_ID', 'nunique')).reset_index()
            master = pd.merge(master, link_agg, on='COMPANY_ID', how='left')
            
        # 심사 코멘트 리스크 플래그 추출
        if 'PAY_BNPL_COMMENTS.csv' in files:
            df_comm = pd.read_csv(os.path.join(root, 'PAY_BNPL_COMMENTS.csv'), dtype={'COMPANY_ID': str})
            df_comm['COMPANY_ID'] = df_comm['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            df_comm['Risk_Flag'] = df_comm['CONTENT'].astype(str).str.contains('거절|불가|보류|주의').astype(int)
            comm_agg = df_comm.groupby('COMPANY_ID')['Risk_Flag'].sum().reset_index(name='NEGATIVE_COMMENT_COUNT')
            master = pd.merge(master, comm_agg, on='COMPANY_ID', how='left')

        # 증빙 서류 제출 충실도
        if 'PAY_BNPL_UPLOAD_FILES.csv' in files:
            df_up = pd.read_csv(os.path.join(root, 'PAY_BNPL_UPLOAD_FILES.csv'), dtype={'COMPANY_ID': str})
            df_up['COMPANY_ID'] = df_up['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            up_agg = df_up.groupby('COMPANY_ID').size().reset_index(name='UPLOADED_FILE_COUNT')
            master = pd.merge(master, up_agg, on='COMPANY_ID', how='left')

    # 5. 최종 클렌징 (결측치는 거래/이력이 없는 것이므로 0으로 보정)
    fill_zero_cols = ['REP_CHANGE_COUNT', 'LINKED_PARTNERS', 'NEGATIVE_COMMENT_COUNT', 'UPLOADED_FILE_COUNT']
    for c in fill_zero_cols:
        if c in master.columns:
            master[c] = master[c].fillna(0)

    output_file = "Final_Integrated_Credit_Master_v2_Complete.csv"
    master.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"\n🎉 [완전체 통합 성공] {len(master)}개 기업, 총 {master.shape[1]}개 심사 Feature 생성 완료!")
    print("\n📊 [완전체 와이드 테이블 샘플 (Top 3)]")
    # 화면 출력 시 가독성을 위해 일부 주요 컬럼만 표시
    display_cols = [c for c in master.columns if c in ['COMPANY_ID', 'LIQUIDITY_RATIO', 'EMPLOYEE_COUNT', 'REP_CHANGE_COUNT', 'LINKED_PARTNERS', 'NEGATIVE_COMMENT_COUNT']]
    print(master[display_cols].head(3).to_string(index=False))
    
    return master

# 즉시 실행
df_ultimate_master = build_ultimate_wide_table('.')

🏗️ [Ultimate Engine] 276홀딩스 완전체 와이드 테이블 통합 가동...
------------------------------------------------------------------------------------------
▶️ 재무 비율(Financial Ratio) 매핑 및 결합 중...
▶️ 조직/경영진 안정성 지표 결합 중...
▶️ 심사 신뢰도 및 네트워크 리스크 압축 중...

🎉 [완전체 통합 성공] 1514개 기업, 총 16개 심사 Feature 생성 완료!

📊 [완전체 와이드 테이블 샘플 (Top 3)]
COMPANY_ID  LIQUIDITY_RATIO  EMPLOYEE_COUNT  REP_CHANGE_COUNT  LINKED_PARTNERS  NEGATIVE_COMMENT_COUNT
0000000013              NaN           160.0               1.0              3.0                     0.0
0000000015            -2.47            18.0               1.0              3.0                     0.0
0000000016              NaN            97.0               1.0              4.0                     0.0
